In [ ]:
import mysql.connector
import pandas as pd
import json
import os

from IPython.display import display
from sshtunnel import SSHTunnelForwarder

Connect to MySQL

Establish connection to the main database and define query function for SQL queries.

In [ ]:
with open('db_config.json') as f:
    DB_CFG = json.load(f)

tunnel = SSHTunnelForwarder((DB_CFG["ssh_host"], int(DB_CFG["ssh_port"])), ssh_username=DB_CFG["ssh_user"], ssh_pkey=os.path.expanduser(DB_CFG["ssh_pkey"]), remote_bind_address=("127.0.0.1", 3306))
tunnel.start()
mydb = mysql.connector.connect(host="127.0.0.1", port=tunnel.local_bind_port, user=DB_CFG["db_user"], database=DB_CFG.get("db_name", "nml"))

def query(sql):
    """Query database with SQL syntax as a string (or fstring)."""
    cursor = mydb.cursor()
    cursor.execute(sql)
    return(cursor.fetchall())

query('SHOW TABLES')

`DRY_RUN = True` reports the rows that would be appended and the existing rows that disagree with the workbook, and writes nothing. After checking that report, set `DRY_RUN = False` and re-run from here: missing rows are appended and conflicting rows are overwritten with the workbook's figures.

`HOURLY_REPLACEMENTS` loads a venue's hourly sheet from a different workbook, for example an earlier Freedom of Information response, in place of the one in `VISITOR_DATA`. The replacement still goes through the hourly sheet check below.

In [ ]:
DRY_RUN = True

VISITOR_DATA = r'data/FOI No.26.38 - data.xlsx'

HOURLY_SHEETS = {
    'WM':   '2.WM-Hourly',
    'WAG':  '3.WAG-Hourly',
    'LLAG': '4.LLAG-Hourly',
    'SH':   '5.SH-Hourly',
    'MoL':  '6.MoL-Hourly',
    'MM':   '7.MM-Hourly',
    'ISM':  '8.ISM-Hourly',
    'PMH':  '9.PMH-Hourly',
    'HH':   '10.HH-Hourly',
}

# venue_id: (workbook path, sheet name). For example, WM's hourly data from the February response:
# HOURLY_REPLACEMENTS = {'WM': (r'data/<file name>.xlsx', '2.WM-Hourly')}
HOURLY_REPLACEMENTS = {'WM': (r'data/FOI.26.09 - data.xlsx', '2.WM-Hourly')}
# An hourly sheet whose total is more than this far from the venue's own daily total is not used.
HOURLY_TOLERANCE = 0.20

hour_map = {
    '10AM': 10, '11AM': 11, '12PM': 12,
    '1PM': 13,  '2PM': 14,  '3PM': 15,
    '4PM': 16,  '5PM': 17
}

Get Visitor Data from Excel

In [ ]:
vdf = pd.read_excel(VISITOR_DATA, '1.NML-Daily')
FY_df = pd.read_excel(VISITOR_DATA, sheet_name='YearStats')

hourly_dfs = {}
for venue_id, sheet in HOURLY_SHEETS.items():
    path, name = HOURLY_REPLACEMENTS.get(venue_id, (VISITOR_DATA, sheet))
    hourly_dfs[venue_id] = pd.read_excel(path, sheet_name=name)
    if venue_id in HOURLY_REPLACEMENTS:
        print(f"{venue_id}: hourly data taken from {os.path.basename(path)}, sheet {name}")

Check the hourly sheets

Each venue's hourly sheet is compared with its own column in `1.NML-Daily` over the days both cover. Small margin of error due to NML daily total methodology. A sheet more than 20% off is not used. no hourly rows are imported for that venue, and its daily visits come from `1.NML-Daily` instead. Sheets that are exact copies of another venue's sheet are also reported.

This check was added after the September response's `2.WM-Hourly` sheet turned out to be an exact copy of `3.WAG-Hourly`.

In [ ]:
_hcols = list(hour_map)
_daily = vdf.set_index('Date')

def _hourly_frame(df):
    return df.set_index('Date')[[c for c in _hcols if c in df.columns]]

sheet_check = {}
untrusted = []
for v, df in hourly_dfs.items():
    h = _hourly_frame(df)
    hs = h.sum(axis=1, min_count=1)
    dv = _daily[v].reindex(hs.index)
    both = hs.notna() & dv.notna()
    ratio = hs[both].sum() / dv[both].sum() if dv[both].sum() else float('nan')
    copies = [w for w, other in hourly_dfs.items()
              if w != v and _hourly_frame(other).reindex(h.index).fillna(-1).equals(h.fillna(-1))]
    ok = abs(ratio - 1) <= HOURLY_TOLERANCE
    sheet_check[v] = {'days_compared': int(both.sum()), 'hourly_to_daily': round(ratio, 3),
                      'identical_to': ', '.join(copies), 'used': ok}
    if not ok:
        untrusted.append(v)

display(pd.DataFrame(sheet_check).T)
for v in untrusted:
    print(f"Not importing hourly rows for {v}: its hourly total is {sheet_check[v]['hourly_to_daily']:.0%} "
          f"of its daily total"
          + (f" (the sheet is identical to {sheet_check[v]['identical_to']})" if sheet_check[v]['identical_to'] else "")
          + ". Its daily visits come from 1.NML-Daily.")
    hourly_dfs.pop(v)

Build Dataframes

Create df for hourly visits per venue, then combine them into a df for daily figures and add any published days outside of the hour range to the daily df, then build financial year df from daily df.

In [ ]:
SOURCES = json.dumps(["National Museums Liverpool"])
DERIVED_HOURLY = json.dumps(["Derived from visits_hourly"])
DERIVED_DAILY = json.dumps(["Derived from visits_daily"])

# Hourly
rows = []
for venue_id, df in hourly_dfs.items():
    for _, row in df.iterrows():
        for hour_col, hour in hour_map.items():
            if hour_col in df.columns and pd.notna(row[hour_col]):
                dt = pd.Timestamp(row['Date']).replace(hour=hour)
                rows.append((dt, venue_id, int(row[hour_col]), SOURCES))

hourly = pd.DataFrame(rows, columns=['datetime', 'venue_id', 'visits', 'sources'])

hourly_day = (hourly.assign(date=hourly['datetime'].dt.normalize())
              .groupby(['date', 'venue_id'])['visits']
              .agg(visits='sum', hours_present='count')
              .reset_index())

# Daily
venue_cols = ['WM', 'WAG', 'MM', 'LLAG', 'SH', 'PMH', 'NCC', 'MoL', 'ISM', 'HH']

melted = vdf[['Date'] + venue_cols].melt(
    id_vars='Date',
    var_name='venue_id',
    value_name='visits'
)
melted = melted.dropna(subset=['visits'])
melted['visits'] = melted['visits'].astype(int)
melted['date'] = pd.to_datetime(melted['Date'])

covered = hourly_day.set_index(['date', 'venue_id']).index
uncovered = melted[~melted.set_index(['date', 'venue_id']).index.isin(covered)]

daily = pd.concat([
    hourly_day[['date', 'venue_id', 'visits']].assign(sources=DERIVED_HOURLY),
    uncovered[['date', 'venue_id', 'visits']].assign(sources=SOURCES)
], ignore_index=True).sort_values(['date', 'venue_id'], ignore_index=True)

# Financial Year
fy_venues = ['WM', 'WAG', 'MM', 'LLAG', 'SH', 'MLL', 'NCC', 'PMH',
             'GWR', 'LOC', 'MLH', 'MoL', 'ISM']

fy_melted = FY_df[['Year'] + fy_venues].melt(
    id_vars='Year',
    var_name='venue_id',
    value_name='visits'
)
fy_melted = fy_melted.dropna(subset=['Year', 'visits'])
fy_melted['visits'] = (fy_melted['visits'].astype(str)
                       .str.replace(',', '', regex=False).astype(float).astype(int))

# everything from the start year, ignoring the sheet's label format
fy_melted['start_year'] = fy_melted['Year'].astype(str).str[:4].astype(int)
fy_melted['fy']       = fy_melted['start_year'].map(lambda y: f"{y}/{(y + 1) % 100:02d}")
fy_melted['fy_start'] = pd.to_datetime(fy_melted['start_year'].map(lambda y: f"{y}-04-01"))
fy_melted['fy_end']   = pd.to_datetime(fy_melted['start_year'].map(lambda y: f"{y + 1}-03-31"))

daily['start_year'] = daily['date'].dt.year - (daily['date'].dt.month < 4)
daily_fy = (daily.groupby(['start_year', 'venue_id'])['visits']
            .sum().rename('daily_total').reset_index())
coverage = (daily.groupby('venue_id')['date']
            .agg(data_min='min', data_max='max').reset_index())

fy = (fy_melted.merge(daily_fy, on=['start_year', 'venue_id'], how='left')
               .merge(coverage, on='venue_id', how='left'))

full_span = (fy['data_min'].notna()
             & (fy['data_min'] <= fy['fy_start'])
             & (fy['data_max'] >= fy['fy_end'])
             & fy['daily_total'].notna())

fy['visits'] = fy['daily_total'].where(full_span, fy['visits']).astype(int)
fy['sources'] = full_span.map({True: DERIVED_DAILY, False: SOURCES})

print(f"Hourly rows {len(hourly)}, daily rows {len(daily)}, financial year rows {len(fy)}")

Compare with the database

Missing rows are those in the workbook but not in the database. Conflicts are existing rows where the workbook and the database disagree. Rows in the database that the workbook does not contain are counted but never removed.

In [ ]:
mydb.database = 'nml'

hourly_db = pd.DataFrame(
    query("SELECT datetime, venue_id, visits FROM visits_hourly"),
    columns=['datetime', 'venue_id', 'db_visits']
)
hourly_db['datetime'] = pd.to_datetime(hourly_db['datetime'])

daily_db = pd.DataFrame(
    query("SELECT date, venue_id, visits FROM visits_daily"),
    columns=['date', 'venue_id', 'db_visits']
)
daily_db['date'] = pd.to_datetime(daily_db['date'])

fy_db = pd.DataFrame(
    query("SELECT fy, venue_id, visits FROM visits_fy"),
    columns=['fy', 'venue_id', 'db_visits']
)
year_db = [r[0] for r in query("SELECT fy FROM financial_year")]

hourly_chk = hourly.merge(hourly_db, on=['datetime', 'venue_id'], how='left')
daily_chk = daily.merge(daily_db, on=['date', 'venue_id'], how='left')
fy_chk = fy.merge(fy_db, on=['fy', 'venue_id'], how='left')

missing_hourly = hourly_chk[hourly_chk['db_visits'].isna()]
missing_daily = daily_chk[daily_chk['db_visits'].isna()]
missing_fy = fy_chk[fy_chk['db_visits'].isna()]
missing_years = (fy[~fy['fy'].isin(year_db)][['fy', 'fy_start', 'fy_end']]
                 .drop_duplicates())

hourly_cf = hourly_chk[hourly_chk['db_visits'].notna()
                       & (hourly_chk['db_visits'] != hourly_chk['visits'])]
daily_cf = daily_chk[daily_chk['db_visits'].notna()
                     & (daily_chk['db_visits'] != daily_chk['visits'])]
fy_cf = fy_chk[fy_chk['db_visits'].notna()
               & (fy_chk['db_visits'] != fy_chk['visits'])]

def _conflict_summary(cf, key):
    return (cf.assign(diff=cf['visits'] - cf['db_visits'])
              .groupby('venue_id')
              .agg(rows=(key, 'count'),
                   total_diff=('diff', 'sum'),
                   worst=('diff', lambda s: s.abs().max()),
                   first=(key, 'min'),
                   last=(key, 'max')))

only_hourly = (hourly_db.merge(hourly[['datetime', 'venue_id']], how='left', indicator=True)
                         .query("_merge == 'left_only'"))
only_daily = (daily_db.merge(daily[['date', 'venue_id']], how='left', indicator=True)
                      .query("_merge == 'left_only'"))
only_in_db = {'hourly': len(only_hourly), 'daily': len(only_daily)}

print(f"HOURLY  {len(missing_hourly)} missing, {len(hourly_cf)} conflicting, "
      f"{len(hourly_chk) - len(missing_hourly) - len(hourly_cf)} matching, "
      f"{only_in_db['hourly']} in the database only")
if not hourly_cf.empty:
    display(_conflict_summary(hourly_cf, 'datetime'))
if not only_hourly.empty:
    print("Hourly rows in the database that this workbook does not supply (kept as they are):")
    display(only_hourly.groupby('venue_id').agg(rows=('datetime', 'count'),
                                                first=('datetime', 'min'), last=('datetime', 'max')))

print(f"DAILY   {len(missing_daily)} missing, {len(daily_cf)} conflicting, "
      f"{len(daily_chk) - len(missing_daily) - len(daily_cf)} matching, "
      f"{only_in_db['daily']} in the database only")
if not daily_cf.empty:
    display(_conflict_summary(daily_cf, 'date'))
if not only_daily.empty:
    print("Daily rows in the database that this workbook does not supply (kept as they are):")
    display(only_daily.groupby('venue_id').agg(rows=('date', 'count'),
                                               first=('date', 'min'), last=('date', 'max')))

print(f"FINANCIAL YEAR  {len(missing_fy)} missing, {len(fy_cf)} conflicting, "
      f"{len(missing_years)} new financial years")
if not fy_cf.empty:
    display(fy_cf.assign(diff=fy_cf['visits'] - fy_cf['db_visits'])
                 [['fy', 'venue_id', 'db_visits', 'visits', 'diff', 'sources']]
                 .sort_values(['fy', 'venue_id']))

Write to the database

Only runs with `DRY_RUN = False`. Missing rows are appended and conflicting rows are overwritten with the workbook's figures and sources.

In [ ]:
def _run(sql, params):
    cursor = mydb.cursor()
    cursor.executemany(sql, params)
    mydb.commit()
    cursor.close()
    return len(params)

if DRY_RUN:
    print("DRY_RUN: nothing written. Check the report above, then set DRY_RUN = False and re-run from the settings cell.")
else:
    # Financial years first, since visits_fy refers to them
    n_years = _run("INSERT INTO financial_year (fy, fy_start, fy_end) VALUES (%s, %s, %s)",
                   [(f, s.date(), e.date()) for f, s, e in missing_years.itertuples(index=False, name=None)])

    n_hourly = _run("INSERT INTO visits_hourly (datetime, venue_id, visits, sources) VALUES (%s, %s, %s, %s)",
                    [(dt.to_pydatetime(), v, int(n), s) for dt, v, n, s
                     in missing_hourly[['datetime', 'venue_id', 'visits', 'sources']].itertuples(index=False, name=None)])
    n_daily = _run("INSERT INTO visits_daily (date, venue_id, visits, sources) VALUES (%s, %s, %s, %s)",
                   [(d.date(), v, int(n), s) for d, v, n, s
                    in missing_daily[['date', 'venue_id', 'visits', 'sources']].itertuples(index=False, name=None)])
    n_fy = _run("INSERT INTO visits_fy (fy, venue_id, visits, sources) VALUES (%s, %s, %s, %s)",
                [(f, v, int(n), s) for f, v, n, s
                 in missing_fy[['fy', 'venue_id', 'visits', 'sources']].itertuples(index=False, name=None)])

    u_hourly = _run("UPDATE visits_hourly SET visits = %s, sources = %s WHERE datetime = %s AND venue_id = %s",
                    [(int(n), s, dt.to_pydatetime(), v) for dt, v, n, s
                     in hourly_cf[['datetime', 'venue_id', 'visits', 'sources']].itertuples(index=False, name=None)])
    u_daily = _run("UPDATE visits_daily SET visits = %s, sources = %s WHERE date = %s AND venue_id = %s",
                   [(int(n), s, d.date(), v) for d, v, n, s
                    in daily_cf[['date', 'venue_id', 'visits', 'sources']].itertuples(index=False, name=None)])
    u_fy = _run("UPDATE visits_fy SET visits = %s, sources = %s WHERE fy = %s AND venue_id = %s",
                [(int(n), s, f, v) for f, v, n, s
                 in fy_cf[['fy', 'venue_id', 'visits', 'sources']].itertuples(index=False, name=None)])

    print(f"Financial years added: {n_years}")
    print(f"Hourly rows added: {n_hourly}, overwritten: {u_hourly}")
    print(f"Daily rows added: {n_daily}, overwritten: {u_daily}")
    print(f"Financial year rows added: {n_fy}, overwritten: {u_fy}")